# CarSathi AI — 05 Model Training

Feature selection, train/test split, preprocessing, baseline, Linear Regression, Decision Tree, and Random Forest training.


## Prerequisites / Setup


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_PATH = Path(
    r"C:\Users\lenovo\Document_local\Data Science NareshIT Course"
    r"\Projects For GITHUB\CarSathi-AI\data\processed\cars_ml.csv"
)

cars_ml = pd.read_csv(PROCESSED_PATH)

# Recreate the feature used by the original notebook.
CURRENT_YEAR = 2025
cars_ml["car_age"] = CURRENT_YEAR - cars_ml["year"]

print("Processed ML dataset loaded.")
print("Shape:", cars_ml.shape)
print("Columns:", cars_ml.columns.tolist())


# The original notebook creates car_age before model training.
# The training cells below will define X/y and the preprocessing pipeline.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ----------------------------------------
# 1. Features
# ----------------------------------------

numerical_features = [
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
    "torque_nm",
    "car_age"
]

categorical_features = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

features = numerical_features + categorical_features

X = cars_ml[features]
y = cars_ml["selling_price"]


# ----------------------------------------
# 2. Train-Test Split
# ----------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ----------------------------------------
# 3. Preprocessor
# ----------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


# ----------------------------------------
# 4. Fit on Train / Transform Both
# ----------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# ----------------------------------------
# 5. Check
# ----------------------------------------

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


print("\n" + "=" * 60)
print("PREPROCESSING")
print("=" * 60)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)


feature_names = preprocessor.get_feature_names_out()

print("\nNumber of processed features:", len(feature_names))

print("\nProcessed Features:")
for feature in feature_names:
    print(feature)

In [ ]:
#Baseline Model

In [ ]:
# Create Baseline Prediction

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Mean selling price from TRAINING data
baseline_prediction = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)

print("Baseline prediction:", y_train.mean())

In [ ]:
# Evaluate Baseline

In [ ]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print("\n" + "=" * 60)
print("BASELINE MODEL RESULTS")
print("=" * 60)

print(f"MAE  : ₹{baseline_mae:,.2f}")
print(f"RMSE : ₹{baseline_rmse:,.2f}")
print(f"R²   : {baseline_r2:.4f}")

In [ ]:
# Create Your Model Comparison Table

In [ ]:
model_results = []

model_results.append({
    "Model": "Baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
})

import pandas as pd

results_df = pd.DataFrame(model_results)

print(results_df)

In [ ]:
# Train Linear Regression

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Fill missing values with median column values
imputer = SimpleImputer(strategy='median')
X_train_processed = imputer.fit_transform(X_train_processed)

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train_processed, y_train)

print("Linear Regression model trained successfully!")

In [ ]:
# Make predictions

In [ ]:
# Apply the fitted imputer to X_test_processed
X_test_processed = imputer.transform(X_test_processed)

# Make predictions
y_pred_lr = lr_model.predict(X_test_processed)

print("First 10 Actual Prices:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Prices:")
print(y_pred_lr[:10])

In [ ]:
# Run evaluation code

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Evaluation metrics
mae_lr = mean_absolute_error(y_test, y_pred_lr)

mse_lr = mean_squared_error(y_test, y_pred_lr)

rmse_lr = np.sqrt(mse_lr)

r2_lr = r2_score(y_test, y_pred_lr)

print("=" * 50)
print("LINEAR REGRESSION RESULTS")
print("=" * 50)

print(f"MAE  : ₹{mae_lr:,.2f}")
print(f"MSE  : {mse_lr:,.2f}")
print(f"RMSE : ₹{rmse_lr:,.2f}")
print(f"R²   : {r2_lr:.4f}")

In [ ]:
# Also calculate percentage error

In [ ]:
mean_price = y_test.mean()

mae_percentage = (mae_lr / mean_price) * 100

print(f"MAE as % of average test price: {mae_percentage:.2f}%")

In [ ]:
# Actual vs Predicted

In [ ]:
comparison_lr = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": y_pred_lr
})

comparison_lr["Error"] = (
    comparison_lr["Actual_Price"]
    - comparison_lr["Predicted_Price"]
)

comparison_lr["Absolute_Error"] = (
    comparison_lr["Error"].abs()
)

print(comparison_lr.head(20))

In [ ]:
# Find the biggest prediction mistakes

In [ ]:
comparison_lr.sort_values(
    "Absolute_Error",
    ascending=False
).head(20)

In [ ]:
# Actual vs Predicted Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred_lr,
    alpha=0.5
)

plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Linear Regression: Actual vs Predicted")

plt.show()

In [ ]:
# Residual Analysis

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_pred_lr,
    comparison_lr["Error"],
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Selling Price")
plt.ylabel("Residual")
plt.title("Linear Regression: Residual Plot")

plt.show()

In [ ]:
# Save the first model results

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [mae_lr],
    "RMSE": [rmse_lr],
    "R2": [r2_lr]
})

print(results)

In [ ]:
# Decision Tree Regression

In [ ]:
# Import Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    random_state=42
)

dt_model.fit(X_train_processed, y_train)

print("Decision Tree model trained successfully!")

In [ ]:
# Predict

In [ ]:
y_pred_dt = dt_model.predict(X_test_processed)

In [ ]:
# Evalute 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("=" * 50)
print("DECISION TREE REGRESSION RESULTS")
print("=" * 50)

print(f"MAE  : ₹{mae_dt:,.2f}")
print(f"MSE  : {mse_dt:,.2f}")
print(f"RMSE : ₹{rmse_dt:,.2f}")
print(f"R²   : {r2_dt:.4f}")

In [ ]:
results.loc[len(results)] = [
    "Decision Tree",
    mae_dt,
    rmse_dt,
    r2_dt
]

print(results)

In [ ]:
# Random Forest Regression

In [ ]:
# Train the model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Make Pridection

In [ ]:
y_pred_rf = rf_model.predict(X_test_processed)

In [ ]:
# Evalute 

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)

mse_rf = mean_squared_error(y_test, y_pred_rf)

rmse_rf = np.sqrt(mse_rf)

r2_rf = r2_score(y_test, y_pred_rf)

print("=" * 50)
print("RANDOM FOREST REGRESSION RESULTS")
print("=" * 50)

print(f"MAE  : ₹{mae_rf:,.2f}")
print(f"MSE  : {mse_rf:,.2f}")
print(f"RMSE : ₹{rmse_rf:,.2f}")
print(f"R²   : {r2_rf:.4f}")

In [ ]:
# Add Random Forest to your comparison

In [ ]:
results.loc[len(results)] = [
    "Random Forest",
    mae_rf,
    rmse_rf,
    r2_rf
]

print(results)